# 🧠 Macro Analyzer & Mood Engine

This notebook simulates the long-term music analysis process (Phase 3). Instead of just looking at the current frame's BPM or power, we collect 1-second snapshots of the audio features into a `MacroLogger` buffer.

We then derive "DJ Journey" metrics like:
- **Power Volatility**
- **Bass vs Treble Ratio Slope** (Build-up detection)
- **General Mood** (Chill, Peak, Buildup, Hypnotic)


In [1]:
import sys
import os
sys.path.append(os.path.abspath('../..'))

import librosa
import numpy as np
import time
from collections import deque
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

import core.Listener as ListenerModule
from IPython.display import display, clear_output

def default_infos():
    return {
        "startServer"     : False ,
        "useMicrophone"   : True  ,
        "HARDWARE_MODE"   : "simulation",
        "onRaspberry"     : False  ,
        "printAsservmentDetails" : False 
    }


In [2]:
# Reusing the Robust Simulated Microphone from ContinuousHybridTracker
class Robust_Simulated_Microphone:
    def __init__(self, y_full_array, bandValues, infos):
        self.bandValues = bandValues
        self.nb_of_fft_band = len(self.bandValues)
        
        self.sample_rate = 44100
        self.buffer_size = 1024 
        self.audio_data = np.zeros(self.buffer_size)
        
        self.full_audio = y_full_array
        self.total_samples = len(self.full_audio)
        self.current_pos = 0
        
        fft_size = self.buffer_size // 2 + 1
        self.weight_matrix = np.zeros((self.nb_of_fft_band, fft_size))
        
        def hz_to_mel(f): return 2595 * np.log10(1 + f / 700.0)
        def mel_to_hz(m): return 700 * (10**(m / 2595.0) - 1)
        
        lower_mel = hz_to_mel(20)
        upper_mel = hz_to_mel(20000)
        mel_points = np.linspace(lower_mel, upper_mel, self.nb_of_fft_band + 2)
        hz_points = mel_to_hz(mel_points)
        bin_points = np.floor((self.buffer_size + 1) * hz_points / self.sample_rate).astype(int)
        
        for i in range(self.nb_of_fft_band):
            start = min(bin_points[i], fft_size - 1)
            mid = min(bin_points[i + 1], fft_size - 1)
            end = min(bin_points[i + 2], fft_size - 1)
            if mid > start:
                self.weight_matrix[i, start:mid] = np.linspace(0, 1, mid - start, endpoint=False)
            if end > mid:
                self.weight_matrix[i, mid:end] = np.linspace(1, 0, end - mid, endpoint=False)
            band_sum = np.sum(self.weight_matrix[i, :])
            if band_sum > 0:
                self.weight_matrix[i, :] /= band_sum
                
        self.raw_fft_history = None
        self.bass_flux = 0.0
        self.treble_flux = 0.0

    def pop_chunk(self, chunk_size=1024):
        if self.current_pos + chunk_size > self.total_samples:
            return False 
        
        incoming = self.full_audio[self.current_pos : self.current_pos + chunk_size]
        self.current_pos += chunk_size
        self.audio_data = np.roll(self.audio_data, -chunk_size)
        self.audio_data[-chunk_size:] = incoming
        return True

    def calculate_fft(self):
        windowed_data = self.audio_data * np.hanning(self.buffer_size)
        fft_result = np.abs(np.fft.rfft(windowed_data))
        
        scale = 150.0 / (self.buffer_size / 1024.0)
        mel_bands = np.dot(self.weight_matrix, fft_result) * scale
        for i in range(self.nb_of_fft_band):
            self.bandValues[i] = int(mel_bands[i])

        if self.raw_fft_history is not None:
            freq_diff = np.maximum(0, fft_result - self.raw_fft_history)
            self.bass_flux = np.sum(freq_diff[1:7])
            self.treble_flux = np.sum(freq_diff[70:300])
        self.raw_fft_history = fft_result


In [3]:
song_files = [
    '../mp3_files/Palladium.mp3',
    '../mp3_files/Pumped Up Kicks.mp3',
    #'../mp3_files/Nobody Rules the Streets.mp3',
    '../mp3_files/Sugar (feat. Francesco Yates).mp3'
    # Add as many songs as you want here!
]

print(f"Loading {len(song_files)} songs sequentially for Song Change Test...")

y_list = []
onset_list = []
true_beat_times_list = []
current_shift_time = 0.0
tempo_librosa = 120.0 # Default fallback

for i, f in enumerate(song_files):
    print(f"Running non-causal Librosa Beat Track on Song {i+1}...")
    y, sr = librosa.load(f.replace('\\\\', '/'), sr=44100)
    y_list.append(y)
    
    onset = librosa.onset.onset_strength(y=y, sr=sr)
    onset_list.append(onset)
    
    t, bf = librosa.beat.beat_track(onset_envelope=onset, sr=sr)
    if i == 0:
        tempo_librosa = float(t[0]) if hasattr(t, '__len__') else float(t) # Used for first plot line
    
    bt = librosa.frames_to_time(bf, sr=sr)
    true_beat_times_list.append(bt + current_shift_time)
    
    current_shift_time += len(y) / float(sr)

import numpy as np
y_full = np.concatenate(y_list) if y_list else np.array([])
onset_env_full = np.concatenate(onset_list) if onset_list else np.array([])
true_beat_times = np.concatenate(true_beat_times_list) if true_beat_times_list else np.array([])

print(f"✅ Ground Truth Extraction Complete! Main Beats: {len(true_beat_times)}, Sub Beats: Not Extracted\n")


Loading 3 songs sequentially for Song Change Test...
Running non-causal Librosa Beat Track on Song 1...


FileNotFoundError: [Errno 2] No such file or directory: '../mp3_files/Palladium.mp3'

### The MacroLogger Simulator
We run the 60FPS loop, but only save data into the `macro_snapshots` list every 1 second.

In [ ]:
infos = default_infos()
SIMULATED_FPS = 60.0
TIME_PER_FRAME = 1.0 / SIMULATED_FPS
CHUNK_SIZE_FOR_60FPS = int(44100 / SIMULATED_FPS)

class MockTime:
    def __init__(self):
        self.current_time = 0.0
    def time(self):
        return self.current_time
mock_timer = MockTime()
ListenerModule.time.time = mock_timer.time

listener = ListenerModule.Listener(infos)
mic = Robust_Simulated_Microphone(y_full, listener.fft_band_values, infos)

# 10-minute snapshot buffer
macro_snapshots = deque(maxlen=600)

# Accumulators for the 1-second window
sec_power_acc = 0.0
sec_bass_acc = 0.0
sec_treble_acc = 0.0
sec_novelty_acc = 0.0
sec_event_count = 0

audio_time = 0.0
frame = 0

print("Running Simulation...")
while mic.pop_chunk(CHUNK_SIZE_FOR_60FPS):
    mic.calculate_fft()
    listener.update()
    
    # Gather data for 1s aggregation
    sec_power_acc += listener.smoothed_total_power
    sec_bass_acc += np.sum(listener.smoothed_fft_band_values[0:2])
    sec_treble_acc += np.sum(listener.smoothed_fft_band_values[6:8])
    sec_novelty_acc += listener.combined_novelty
    
    if listener.is_verse_chorus_change or listener.is_song_change:
        sec_event_count += 1
        
    # Every 1 second (60 frames)
    if frame % int(SIMULATED_FPS) == 0 and frame > 0:
        snapshot = {
            'time': audio_time,
            'bpm': listener.bpm,
            'bpm_trust': listener.binary_trust,
            'power': sec_power_acc / SIMULATED_FPS,
            'bass_power': sec_bass_acc / SIMULATED_FPS,
            'treble_power': sec_treble_acc / SIMULATED_FPS,
            'novelty': sec_novelty_acc / SIMULATED_FPS,
            'events': sec_event_count
        }
        macro_snapshots.append(snapshot)
        
        # Reset accumulators
        sec_power_acc = 0.0
        sec_bass_acc = 0.0
        sec_treble_acc = 0.0
        sec_novelty_acc = 0.0
        sec_event_count = 0

    audio_time += TIME_PER_FRAME
    mock_timer.current_time += TIME_PER_FRAME
    frame += 1

print(f"Simulation complete. Gathered {len(macro_snapshots)} snapshots.")


### Calculating Derived Metrics
Here we extract 3-minute and 5-minute rolling trends.

In [ ]:
def get_macro_mood(snapshots, window_seconds=180):
    if len(snapshots) < 10:
        return "WARMUP" # Not enough data yet
        
    # Get the last N snapshots
    window = list(snapshots)[-min(len(snapshots), window_seconds):]
    
    powers = np.array([s['power'] for s in window])
    basses = np.array([s['bass_power'] for s in window])
    trebles = np.array([s['treble_power'] for s in window])
    bpms = np.array([s['bpm'] for s in window])
    trusts = np.array([s['bpm_trust'] for s in window])
    events = sum(s['events'] for s in window)
    
    avg_power = np.mean(powers)
    max_power = np.max(powers)
    
    current_power = powers[-1]
    current_bass = basses[-1]
    current_treble = trebles[-1]
    
    avg_trust = np.mean(trusts)
    
    # Calculate Bass-to-Treble slope over the last 30 seconds to detect buildups
    recent_30s = list(snapshots)[-min(len(snapshots), 30):]
    recent_basses = np.array([s['bass_power'] for s in recent_30s])
    recent_trebles = np.array([s['treble_power'] for s in recent_30s])
    
    bass_slope = np.polyfit(np.arange(len(recent_basses)), recent_basses, 1)[0] if len(recent_basses) > 1 else 0
    treble_slope = np.polyfit(np.arange(len(recent_trebles)), recent_trebles, 1)[0] if len(recent_trebles) > 1 else 0
    
    # Mood Logic
    if current_power < avg_power * 0.5 and avg_trust < 0.3:
        return "CHILL"
        
    if bass_slope < -0.1 and treble_slope > 0.05:
        return "BUILDUP"
        
    if current_power > max_power * 0.85 and events > 2:
        return "PEAK"
        
    if events == 0 and current_power > avg_power * 0.7 and avg_trust > 0.6:
        return "HYPNOTIC"
        
    return "GROOVE"

# Apply mood retrospectively to see the journey
mood_journey = []
for i in range(10, len(macro_snapshots)):
    mood_journey.append(get_macro_mood(list(macro_snapshots)[:i]))


In [ ]:
times = [s['time'] for s in macro_snapshots][10:]
powers = [s['power'] for s in macro_snapshots][10:]
basses = [s['bass_power'] for s in macro_snapshots][10:]

plt.figure(figsize=(15, 6))
plt.plot(times, powers, label="Total Power", color="#aaa")
plt.plot(times, basses, label="Bass Power", color="#2196F3")

colors = {"CHILL": "#4dd0e1", "BUILDUP": "#ffb300", "PEAK": "#e53935", "HYPNOTIC": "#8e24aa", "GROOVE": "#43a047", "WARMUP": "#cccccc"}

# Shade regions based on mood
for i in range(len(times) - 1):
    plt.axvspan(times[i], times[i+1], color=colors[mood_journey[i]], alpha=0.3, lw=0)
    
# Create custom legend for moods
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=color, label=mood, alpha=0.5) for mood, color in colors.items()]
legend_elements.append(plt.Line2D([0], [0], color='#aaa', lw=2, label='Total Power'))
legend_elements.append(plt.Line2D([0], [0], color='#2196F3', lw=2, label='Bass Power'))

plt.legend(handles=legend_elements, loc="upper left")
plt.title("DJ Journey: Macro Mood Engine Analysis")
plt.xlabel("Time (seconds)")
plt.ylabel("Amplitude")
plt.tight_layout()
plt.show()
